# Manual multilabel training

The epoch loop is kept here for interactive inspection. Reusable operations live in `src/`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader

In [ ]:
ROOT = Path.cwd().resolve()
if not (ROOT / "configs/framework.yaml").exists():
    ROOT = ROOT.parent
if not (ROOT / "configs/framework.yaml").exists():
    raise FileNotFoundError("Run this notebook from the repository or notebooks directory")
sys.path.insert(0, str(ROOT))

from src.data import fit_standardizer, make_dataset, split_indices
from src.hashing import calculate_run_id
from src.model import MultilabelMLP
from src.training import (
    build_loss,
    build_optimizer,
    evaluate_epoch,
    save_artifacts,
    train_epoch,
)

In [ ]:
config_path = ROOT / "configs/framework.yaml"
config_snapshot = config_path.read_bytes()
config = yaml.safe_load(config_snapshot)
run_id = calculate_run_id(config)
print(f"run_id: {run_id}")

In [ ]:
data_path = ROOT / config["data"]["path"]
frame = pd.read_parquet(data_path)
feature_columns = config["data"]["feature_columns"]
label_columns = config["data"]["label_columns"]
if not feature_columns or not label_columns:
    raise ValueError("Set data.feature_columns and data.label_columns in framework.yaml")
missing_columns = set(feature_columns + label_columns) - set(frame.columns)
if missing_columns:
    raise ValueError(f"Columns not found in dataset: {sorted(missing_columns)}")
frame[feature_columns + label_columns].head()

In [ ]:
split = config["split"]
train_idx, val_idx, test_idx = split_indices(
    len(frame), split["train"], split["val"], split["test"], split["seed"]
)
if min(map(len, (train_idx, val_idx, test_idx))) == 0:
    raise ValueError("Dataset is too small for non-empty train, validation, and test splits")

means, scales = fit_standardizer(frame.iloc[train_idx], feature_columns)
train_data = make_dataset(frame.iloc[train_idx], feature_columns, label_columns, means, scales)
val_data = make_dataset(frame.iloc[val_idx], feature_columns, label_columns, means, scales)
test_data = make_dataset(frame.iloc[test_idx], feature_columns, label_columns, means, scales)

batch_size = config["training"]["batch_size"]
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size)
test_loader = DataLoader(test_data, batch_size=batch_size)
len(train_data), len(val_data), len(test_data)

In [ ]:
torch.manual_seed(split["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultilabelMLP(
    num_features=len(feature_columns),
    num_labels=len(label_columns),
    **config["model"],
).to(device)
criterion = build_loss(config["loss"]).to(device)
optimizer = build_optimizer(model, config["training"])
print(model)
print(f"device: {device}, loss: {criterion.__class__.__name__}")

Run the next cell to train. Interrupt between epochs if you want to inspect `model` or a batch manually.

In [ ]:
history = []
for epoch in range(1, config["training"]["epochs"] + 1):
    train_metrics = train_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate_epoch(model, val_loader, criterion, device)
    row = {
        "epoch": epoch,
        **{f"train_{key}": value for key, value in train_metrics.items()},
        **{f"val_{key}": value for key, value in val_metrics.items()},
    }
    history.append(row)
    print(
        f"{epoch:03d} | train loss {train_metrics['loss']:.4f} "
        f"f1 {train_metrics['micro_f1']:.3f} | val loss {val_metrics['loss']:.4f} "
        f"f1 {val_metrics['micro_f1']:.3f} acc {val_metrics['accuracy']:.3f}"
    )

In [ ]:
test_metrics = evaluate_epoch(model, test_loader, criterion, device)
metrics = {
    "run_id": run_id,
    "final_train": train_metrics,
    "final_validation": val_metrics,
    "test": test_metrics,
}
output_dir = save_artifacts(
    run_id, model, config_snapshot, metrics, history, ROOT / "artifacts"
)
print(f"Saved artifacts to {output_dir}")
metrics